# MNIST Keras 3 Walkthrough

This notebook mirrors the Keras 3 cell sequence from the MNIST Python chapter. Use it for interactive learning: run one cell at a time, inspect the printed shapes, and compare predictions with the true labels.

For repeatable command-line runs, backend checks, and smoke tests, use `mnist_keras3.py` in this directory.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

## 1. Select the Backend and Import Libraries

Choose the Keras backend before importing Keras. If you change the backend after running this cell, restart the kernel before running the notebook again.

In [ ]:
import os
import platform
import time

# External KERAS_BACKEND wins. Default is TensorFlow for convenience.
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers

## 2. Set Constants and Print Run Details

Keep the seed, batch size, epoch count, Python version, Keras version, and backend in the run record.

In [ ]:
SEED = 1234
BATCH_SIZE = 128
EPOCHS = 5

keras.utils.set_random_seed(SEED)

print("Python:", platform.python_version())
print("Keras:", keras.__version__)
print("Backend:", keras.backend.backend())

## 3. Load, Normalize, and Split MNIST

Before running this cell, predict the shapes of `x_train`, `x_val`, and `x_test`. After running it, explain why the test set is not passed to `fit`.

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = (
    keras.datasets.mnist.load_data()
)

x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = x_train_full[:-10000]
y_train = y_train_full[:-10000]
x_val = x_train_full[-10000:]
y_val = y_train_full[-10000:]

print("train:", x_train.shape, y_train.shape)
print("val:", x_val.shape, y_val.shape)
print("test:", x_test.shape, y_test.shape)

## 4. Define the Two-Hidden-Layer Model

The architecture is `28 x 28 -> 784 -> 256 -> 128 -> 10`. The final layer returns logits, not probabilities.

In [ ]:
model = keras.Sequential(
    [
        layers.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(10),
    ]
)

## 5. Choose the Optimizer, Loss, and Metric

Sparse categorical cross-entropy matches integer labels. `from_logits=True` matches the final layer, which returns raw class scores rather than normalized probabilities. Keep this contract aligned whenever you change the model head.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

model.summary()

## 6. Train, Evaluate, and Inspect Predictions

Use validation metrics during training and reserve the test set for the final check. After evaluation, inspect a few predictions so the reported accuracy is connected to actual images and labels rather than only a scalar metric.

In [ ]:
start = time.perf_counter()
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
)
elapsed = time.perf_counter() - start

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"elapsed_seconds={elapsed:.2f}")
print(f"test_loss={test_loss:.4f} test_accuracy={test_acc:.4f}")

logits = model.predict(x_test[:8], verbose=0)
probs = keras.ops.softmax(logits, axis=-1)
predicted = keras.ops.argmax(probs, axis=-1)
print("predicted:", keras.ops.convert_to_numpy(predicted))
print("true:", y_test[:8])

## What To Inspect Before Moving On

Before adapting this notebook, check that the train, validation, and test shapes match your expectations; that the loss matches the final layer; and that the validation result, final test result, and sampled predictions tell a consistent story. For repeatable smoke checks or saved artifacts, use `mnist_keras3.py` rather than treating notebook state as the record.

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.